# Lighthouse Notebook 2: sentence embeddings and logistic regression

Each of you is building a different model this week on the same data. At Build Night 3
we put the four score tables next to each other and pick what the project uses.

The data is different from last week. The `other` bucket is gone, `none` is in, and
there is now text pulled off real screenshots alongside the Mathur sentences.

The setup cells load the data. How you get from there to a working model is up to you.
You have AI tools available. Use them to write code, not to decide what to do.

Turn on the GPU before you start: Runtime, Change runtime type, T4 GPU.

## Setup

Run these three. Upload `contextdp_ocr.csv` from the shared folder first, using the file
browser on the left.

In [6]:
URL = ("https://raw.githubusercontent.com/aruneshmathur/dark-patterns/"
       "master/data/final-dark-patterns/dark-patterns.csv")

MATHUR_TO_OURS = {
    "Low-stock Message":                "scarcity",
    "High-demand Message":              "scarcity",
    "Countdown Timer":                  "urgency",
    "Limited-time Message":             "urgency",
    "Activity Notification":            "social_proof",
    "Testimonials of Uncertain Origin": "social_proof",
    "Confirmshaming":                   "confirmshaming",
    "Visual Interference":              "hidden_information",
}

import pandas as pd, numpy as np

mathur = pd.read_csv(URL)
mathur["label"] = mathur["Pattern Type"].map(MATHUR_TO_OURS)
mathur = mathur.dropna(subset=["label", "Pattern String"]).copy()
mathur["text"] = mathur["Pattern String"].str.strip()
mathur = mathur[mathur["text"] != ""].drop_duplicates("text")
mathur["site"] = (mathur["Website Page"].astype(str)
                  .str.extract(r"https?://([^/]+)")[0]
                  .str.replace(r"^www\.", "", regex=True))
mathur = mathur.dropna(subset=["site"])[["text", "label", "site"]].reset_index(drop=True)

print(len(mathur), "sentences from", mathur["site"].nunique(), "websites")
print(mathur["label"].value_counts())

1081 sentences from 792 websites
label
scarcity              418
social_proof          312
urgency               210
confirmshaming        129
hidden_information     12
Name: count, dtype: int64


In [7]:
# 440 screenshots from ContextDP, labelled with the mapping we built at Build Night 2,
# with the text already pulled off them by OCR.

shots = pd.read_csv("contextdp_ocr.csv")
print(len(shots), "screenshots")
print(shots["label"].value_counts())
shots.head(3)

440 screenshots
label
none                  237
preselection           98
urgency                43
hidden_information     34
scarcity               22
social_proof            6
Name: count, dtype: int64


,image,platform,their_labels,label,text
0,droom.in.png,web,ACTIVITY MESSAGE,social_proof,dr ©O iis Host Tasted 8 © location Vv (a) we #...
1,kiiroo.com.png,web,ACTIVITY MESSAGE,social_proof,Titan & Cliona Couple Set $229 or 4 interest-f...
2,nutrabay.com.png,web,ACTIVITY MESSAGE,social_proof,NUTRABAY BRANDS SPORTS NUTRITION WEIGHT Loss W...


In [8]:
held_out = pd.DataFrame([
    ("Better grab it, barely any left", "scarcity"),
    ("Buy it before the timer runs out", "urgency"),
    ("There are hardly any of these remaining", "scarcity"),
    ("You would be missing out if you say no", "confirmshaming"),
    ("Everyone is buying this right now", "social_proof"),
    ("People can't stop buying this product", "social_proof"),
    ("The performance of the product is better now, only 17 made ever", "scarcity"),
    ("Are you sure you want to stop protecting your family?", "confirmshaming"),
    ("not a dark pattern, nothing to see here", "none"),
    ("1000 left", "none"),
    ("50% but only for the next decade", "none"),
    ("between 0 and 1 million left in stock", "none"),
    ("20% off but just for today", "urgency"),
    ("Sale ends in 01:59:00", "urgency"),
    ("bought 200 times, only ten left in stock", "scarcity"),
    ("bought 200 times", "social_proof"),
    ("Limited time offer: get 30% off your first 3 months till september 30th. "
     "You can always cancel anytime", "urgency"),
    ("10 days left for our end of summer sale. Get 50% off until the end of the month.",
     "urgency"),
    ("Get our new student discount. 15% off with a valid student ID", "none"),
    ("Rate our Product! Remind me later.", "none"),
    ("Toggled settings to allow notifications and data collection by default",
     "preselection"),
    ("90k people have viewed this item", "social_proof"),
    ("35% off on all items for a limited time only!", "urgency"),
    ("only 1 item left in stock", "scarcity"),
    ("25 people have this in their cart at the moment", "scarcity"),
    ("No thanks, I'd rather pay full price", "confirmshaming"),
    ("Over 12,000 people bought this today", "social_proof"),
    ("Student discounts offered", "none"),
    ("70% of our users are making more than 700K", "social_proof"),
    ("Everyone in dallas is buying the new ford bronco, now is your time", "social_proof"),
    ("The buy one get one free deal will end in 3 hours, shop now", "urgency"),
    ("the new harry potter movie is dropping on 22 september this year", "none"),
    ("Free shipping on orders over $50", "none"),
    ("Currently out of stock", "none"),
    ("4.6 out of 5 stars from 1,204 reviews", "none"),
    ("Summer sale: 20% off all sandals", "none"),
    ("Ships in 3 to 5 business days", "none"),
    ("Return it within 30 days for a full refund", "none"),
    ("Now available in three colors", "none"),
    ("Add to cart", "none"),
    ("In stock at your local store", "none"),
    ("Save this item to your wishlist", "none"),
    ("We use cookies. Accept all or manage your preferences.", "none"),
    ("Compare similar items", "none"),
    ("Price: $24.99, tax calculated at checkout", "none"),
    ("Sign in to see your order history", "none"),
    ("This product weighs 2.3 lbs and ships from Ohio", "none"),
], columns=["text", "label"])

print(len(held_out), "held out sentences,", (held_out.label == "none").sum(), "of them none")

47 held out sentences, 23 of them none


## 1. Split the data

Use these exact settings. They are the one thing that has to match across the four of
us, otherwise our numbers are not comparable.

Sentences: grouped split on the `site` column, 20% test, `random_state=42`. Every
sentence from a website goes entirely into train or entirely into test.

Screenshots: 60/40 split, stratified on label, `random_state=42`.

Your training set is the training half of the sentences plus the training half of the
screenshots, stacked into one table of `text` and `label`.

In [9]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit, train_test_split

# Load the screenshot data.
shots = pd.read_csv("contextdp_ocr.csv")

# Split Mathur sentences by website: 20% of websites in the test set.
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(mathur, groups=mathur["site"])
)

sentence_train = mathur.iloc[train_idx].copy()
sentence_test = mathur.iloc[test_idx].copy()

# Split screenshots: 60% train, 40% test, stratified by label.
shot_train, shot_test = train_test_split(
    shots,
    test_size=0.40,
    stratify=shots["label"],
    random_state=42
)

# Combine the two training sets.
train = pd.concat(
    [
        sentence_train[["text", "label"]],
        shot_train[["text", "label"]]
    ],
    ignore_index=True
)

# Make sure every text value can be passed to the embedding model.
for frame in [train, sentence_test, shot_test]:
    frame["text"] = frame["text"].fillna("").astype(str)

assert set(sentence_train["site"]).isdisjoint(
    set(sentence_test["site"])
)

print("Sentence training rows:", len(sentence_train))
print("Sentence test rows:", len(sentence_test))
print("Screenshot training rows:", len(shot_train))
print("Screenshot test rows:", len(shot_test))
print("Combined training rows:", len(train))
print("\nTraining labels:")
print(train["label"].value_counts())

Sentence training rows: 863
Sentence test rows: 218
Screenshot training rows: 264
Screenshot test rows: 176
Combined training rows: 1127

Training labels:
label
scarcity              347
social_proof          248
urgency               199
none                  142
confirmshaming        101
preselection           59
hidden_information     31
Name: count, dtype: int64


## 2. Build sentence embeddings plus logistic regression

Replace TF-IDF with sentence embeddings, keep logistic regression on top.

TF-IDF counts words, so "only 2 left" and "barely any remaining" look unrelated to it.
An embedding model turns a sentence into numbers based on meaning. You are testing
whether that difference matters for us.

Use a small sentence encoder that runs on the T4. The `sentence-transformers` library is
the easy way in. Anything you predict on has to be encoded the same way the training
text was.

In [10]:
%pip -q install sentence-transformers

import torch
import sklearn
import sentence_transformers

from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression

if not torch.cuda.is_available():
    raise RuntimeError(
        "Enable the T4 GPU under Runtime > Change runtime type, then run again."
    )

np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

encoder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device="cuda"
)

# Use this exact encoding function for training and every evaluation set.
def encode_texts(texts):
    return encoder.encode(
        texts.tolist(),
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

X_train = encode_texts(train["text"])

model = LogisticRegression(
    C=1.0,
    class_weight="balanced",
    solver="lbfgs",
    max_iter=2000,
    random_state=42
)

model.fit(X_train, train["label"])

print("GPU:", torch.cuda.get_device_name(0))
print("Encoder token limit:", encoder.max_seq_length)
print("Training embedding shape:", X_train.shape)
print("Classes:", model.classes_)
print("scikit-learn version:", sklearn.__version__)
print("sentence-transformers version:", sentence_transformers.__version__)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

GPU: Tesla T4
Encoder token limit: 256
Training embedding shape: (1127, 384)
Classes: ['confirmshaming' 'hidden_information' 'none' 'preselection' 'scarcity'
 'social_proof' 'urgency']
scikit-learn version: 1.6.1
sentence-transformers version: 5.7.0


## 3. Look at the OCR text

Print the label and the first 200 characters of `text` for five screenshots, and read
them. You need this for question 3.

In [11]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score
)

eval_sets = {
    "sentences": sentence_test,
    "screenshots": shot_test,
    "held_out": held_out
}

predictions = {}
reports = {}
score_rows = []

for eval_name, frame in eval_sets.items():
    X_eval = encode_texts(frame["text"])
    predicted = model.predict(X_eval)

    predictions[eval_name] = predicted

    print(f"\n===== {eval_name.upper()} =====")
    print(
        classification_report(
            frame["label"],
            predicted,
            digits=3,
            zero_division=0
        )
    )

    reports[eval_name] = classification_report(
        frame["label"],
        predicted,
        output_dict=True,
        zero_division=0
    )

    score_rows.append({
        "model": "embeddings_logreg",
        "eval_set": eval_name,
        "accuracy": accuracy_score(frame["label"], predicted),
        "macro_f1": f1_score(
            frame["label"],
            predicted,
            average="macro",
            zero_division=0
        )
    })

results = pd.DataFrame(
    score_rows,
    columns=["model", "eval_set", "accuracy", "macro_f1"]
)

print("\nScore table:")
print(results.to_string(index=False))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]


===== SENTENCES =====
                    precision    recall  f1-score   support

    confirmshaming      1.000     0.964     0.982        28
hidden_information      1.000     1.000     1.000         1
              none      0.000     0.000     0.000         0
          scarcity      1.000     0.952     0.976        84
      social_proof      1.000     1.000     1.000        68
           urgency      0.902     1.000     0.949        37

          accuracy                          0.977       218
         macro avg      0.817     0.819     0.818       218
      weighted avg      0.983     0.977     0.980       218



Batches:   0%|          | 0/6 [00:00<?, ?it/s]


===== SCREENSHOTS =====
                    precision    recall  f1-score   support

hidden_information      0.250     0.357     0.294        14
              none      0.755     0.747     0.751        95
      preselection      0.667     0.718     0.691        39
          scarcity      1.000     0.111     0.200         9
      social_proof      0.000     0.000     0.000         2
           urgency      0.500     0.412     0.452        17

          accuracy                          0.636       176
         macro avg      0.529     0.391     0.398       176
      weighted avg      0.675     0.636     0.636       176



Batches:   0%|          | 0/2 [00:00<?, ?it/s]


===== HELD_OUT =====
                    precision    recall  f1-score   support

    confirmshaming      1.000     0.667     0.800         3
hidden_information      0.000     0.000     0.000         0
              none      1.000     0.130     0.231        23
      preselection      0.250     1.000     0.400         1
          scarcity      0.400     0.667     0.500         6
      social_proof      0.583     1.000     0.737         7
           urgency      0.538     1.000     0.700         7

          accuracy                          0.511        47
         macro avg      0.539     0.638     0.481        47
      weighted avg      0.777     0.511     0.450        47


Score table:
            model    eval_set  accuracy  macro_f1
embeddings_logreg   sentences  0.977064  0.817691
embeddings_logreg screenshots  0.636364  0.398069
embeddings_logreg    held_out  0.510638  0.481087


## 4. Score it three ways

Print a `classification_report` for each:

1. the sentence test set
2. the screenshot test set
3. `held_out`

In [12]:
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score
)

eval_sets = {
    "sentences": sentence_test,
    "screenshots": shot_test,
    "held_out": held_out
}

predictions = {}
reports = {}
score_rows = []

for eval_name, frame in eval_sets.items():
    X_eval = encode_texts(frame["text"])
    predicted = model.predict(X_eval)

    predictions[eval_name] = predicted

    print(f"\n===== {eval_name.upper()} =====")
    print(
        classification_report(
            frame["label"],
            predicted,
            digits=3,
            zero_division=0
        )
    )

    reports[eval_name] = classification_report(
        frame["label"],
        predicted,
        output_dict=True,
        zero_division=0
    )

    score_rows.append({
        "model": "embeddings_logreg",
        "eval_set": eval_name,
        "accuracy": accuracy_score(frame["label"], predicted),
        "macro_f1": f1_score(
            frame["label"],
            predicted,
            average="macro",
            zero_division=0
        )
    })

results = pd.DataFrame(
    score_rows,
    columns=["model", "eval_set", "accuracy", "macro_f1"]
)

print("\nScore table:")
print(results.to_string(index=False))

Batches:   0%|          | 0/7 [00:00<?, ?it/s]


===== SENTENCES =====
                    precision    recall  f1-score   support

    confirmshaming      1.000     0.964     0.982        28
hidden_information      1.000     1.000     1.000         1
              none      0.000     0.000     0.000         0
          scarcity      1.000     0.952     0.976        84
      social_proof      1.000     1.000     1.000        68
           urgency      0.902     1.000     0.949        37

          accuracy                          0.977       218
         macro avg      0.817     0.819     0.818       218
      weighted avg      0.983     0.977     0.980       218



Batches:   0%|          | 0/6 [00:00<?, ?it/s]


===== SCREENSHOTS =====
                    precision    recall  f1-score   support

hidden_information      0.250     0.357     0.294        14
              none      0.755     0.747     0.751        95
      preselection      0.667     0.718     0.691        39
          scarcity      1.000     0.111     0.200         9
      social_proof      0.000     0.000     0.000         2
           urgency      0.500     0.412     0.452        17

          accuracy                          0.636       176
         macro avg      0.529     0.391     0.398       176
      weighted avg      0.675     0.636     0.636       176



Batches:   0%|          | 0/2 [00:00<?, ?it/s]


===== HELD_OUT =====
                    precision    recall  f1-score   support

    confirmshaming      1.000     0.667     0.800         3
hidden_information      0.000     0.000     0.000         0
              none      1.000     0.130     0.231        23
      preselection      0.250     1.000     0.400         1
          scarcity      0.400     0.667     0.500         6
      social_proof      0.583     1.000     0.737         7
           urgency      0.538     1.000     0.700         7

          accuracy                          0.511        47
         macro avg      0.539     0.638     0.481        47
      weighted avg      0.777     0.511     0.450        47


Score table:
            model    eval_set  accuracy  macro_f1
embeddings_logreg   sentences  0.977064  0.817691
embeddings_logreg screenshots  0.636364  0.398069
embeddings_logreg    held_out  0.510638  0.481087


## 5. Find what it got wrong

Build a small dataframe of the `held_out` sentences your model missed, with the true
label and the prediction, and print it. Bring this to Build Night 3.

In [13]:
held_out_predictions = held_out[["text", "label"]].copy()
held_out_predictions["predicted"] = predictions["held_out"]

wrong = held_out_predictions.loc[
    held_out_predictions["label"] != held_out_predictions["predicted"],
    ["text", "label", "predicted"]
].reset_index(drop=True)

print(f"Missed {len(wrong)} of {len(held_out)} held-out sentences.")

if wrong.empty:
    print("No held-out mistakes.")
else:
    print(wrong.to_string(index=False))

print("\nMistakes where the true label was 'none':")
wrong_none = wrong[wrong["label"] == "none"]

if wrong_none.empty:
    print("None.")
else:
    print(wrong_none.to_string(index=False))

Missed 23 of 47 held-out sentences.
                                                            text          label          predicted
                          You would be missing out if you say no confirmshaming hidden_information
 The performance of the product is better now, only 17 made ever       scarcity       social_proof
                         not a dark pattern, nothing to see here           none       social_proof
                                                       1000 left           none           scarcity
                                50% but only for the next decade           none           scarcity
                           between 0 and 1 million left in stock           none           scarcity
   Get our new student discount. 15% off with a valid student ID           none       preselection
                 25 people have this in their cart at the moment       scarcity       social_proof
                                       Student discounts offered         

## 6. Save three files

Use these names and columns so they merge:

`results_YOURNAME.csv` with columns `model, eval_set, accuracy, macro_f1`, one row per
eval set. Model name for this notebook is `embeddings_logreg`.

`wrong_YOURNAME.csv`, the dataframe from step 5.

`preds_YOURNAME.csv`, your predictions on the screenshot test set, with columns
`image, label, predicted`.

Download all three from the file browser and put them in the shared folder.

In [14]:
YOURNAME = "Prasanna Ramamurthi"

if YOURNAME == "YOURNAME":
    raise ValueError("Replace YOURNAME with your name before saving.")

screenshot_predictions = shot_test[["image", "label"]].copy()
screenshot_predictions["predicted"] = predictions["screenshots"]

files_to_save = {
    f"results_{YOURNAME}.csv": results,
    f"wrong_{YOURNAME}.csv": wrong,
    f"preds_{YOURNAME}.csv": screenshot_predictions
}

for filename, frame in files_to_save.items():
    frame.to_csv(filename, index=False)
    print(f"Saved {filename}: {len(frame)} rows")

Saved results_Prasanna Ramamurthi.csv: 3 rows
Saved wrong_Prasanna Ramamurthi.csv: 23 rows
Saved preds_Prasanna Ramamurthi.csv: 176 rows


## Submitting

1. Runtime, then Restart session and run all. Wait for it to finish.
2. Scroll through and check every cell has its output showing. If a cell is empty, your
   results did not save.
3. File, Download, Download .ipynb.
4. Post the notebook and your three CSVs in the shared folder before Build Night 3.

If something crashes halfway, fix it and run all again. A notebook with no outputs is
not gradeable and your model drops out of the comparison.

## 7. Questions

1. Your three accuracy numbers.
2. Why is the screenshot score different from the sentence score?
3. How is OCR text different from a Mathur sentence? One line.
4. Two held out sentences you got wrong, with what the model guessed. Was one of them a
   `none`?
5. Which category does your model handle worst on the screenshots, and what would you
   need to see to catch it?
6. One thing that confused you.
7. Guess how your model does against the other three. We check next week.